# AF-MOT Football Player Tracking & CVAT Export Pipeline

This notebook processes football video clips to perform player detection, tracking, CVAT MOT 1.1 pre-annotation export, and tracking performance evaluation against ground truth.

### Cell 1 — Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Cell 2 — Set project code path

In [2]:
from pathlib import Path
import os, sys

PROJECT_CODE_DIR = Path('/content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Utilities/Football_MOT_Exporting')
os.chdir(PROJECT_CODE_DIR)
sys.path.insert(0, str(PROJECT_CODE_DIR))
print('Working directory:', Path.cwd())

Working directory: /content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Utilities/Football_MOT_Exporting


### Cell 3 — Install dependencies

In [3]:
!python scripts/install_dependencies.py

Installing: ultralytics opencv-python-headless pandas pyyaml tqdm boxmot scipy
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 7.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.6/676.6 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 854.4/854.4 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 96.0 MB/s eta 0:00:00
  

### Cell 3.1 Consolidated BoxMOT Setup & Patching

This cell performs all necessary steps to install `boxmot` from its GitHub repository in an editable mode and applies all required patches to resolve import errors and API changes. Run this cell to ensure `boxmot` is correctly configured before proceeding to tracking tasks.

In [ ]:
import re
import os
import sys
from pathlib import Path
import subprocess

print("---> Starting Consolidated BoxMOT Setup & Patching <---")

# --- 1. Uninstall existing boxmot and ultralytics for a clean slate ---
print("Uninstalling existing boxmot and ultralytics...")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "boxmot", "ultralytics"], capture_output=True, text=True)
print("Uninstallation complete.")

# --- 2. Clone boxmot repository if it doesn't exist ---
boxmot_repo_path = '/tmp/boxmot_repo'
if not os.path.exists(boxmot_repo_path):
    print(f"Cloning BoxMOT repository to {boxmot_repo_path}...")
    subprocess.run(["git", "clone", "https://github.com/mikel-brostrom/boxmot.git", boxmot_repo_path], check=True, capture_output=True, text=True)
    print("BoxMOT repository cloned.")
else:
    print(f"BoxMOT repository already cloned to {boxmot_repo_path}. Skipping clone.")

# --- 3. Install boxmot in editable mode ---
print(f"Installing boxmot in editable mode from {boxmot_repo_path}...")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", boxmot_repo_path], check=True, capture_output=True, text=True)
print("BoxMOT installed in editable mode.")

# --- 4. Reinstall ultralytics ---
print("Reinstalling ultralytics...")
subprocess.run([sys.executable, "-m", "pip", "install", "ultralytics"], check=True, capture_output=True, text=True)
print("Ultralytics reinstalled.")

# --- 5. Patch `src/tracker.py` for correct `BoTSORT` import path ---
print("Patching `src/tracker.py` import paths...")
PROJECT_CODE_DIR = Path('/content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Utilities/Football_MOT_Exporting')
tracker_file_path = PROJECT_CODE_DIR / 'src' / 'tracker.py'

if not tracker_file_path.exists():
    raise FileNotFoundError(f"Error: {tracker_file_path} not found. Please ensure your project structure is correct.")

with open(tracker_file_path, 'r') as f:
    content_tracker = f.read()

pattern_v20 = r"from boxmot\\.trackers\\.bbox\\.botsort\\.botsort import (BotSort|BoTSORT)"
replacement_v20 = r"from boxmot.trackers.bbox.botsort import \\1"
pattern_legacy = r"from boxmot\\.trackers\\.botsort\\.botsort import (BotSort|BoTSORT|BOTSORT)"
replacement_legacy = r"from boxmot.trackers.bbox.botsort import \\1"

new_content_tracker = re.sub(pattern_v20, replacement_v20, content_tracker)
new_content_tracker = re.sub(pattern_legacy, replacement_legacy, new_content_tracker)

if new_content_tracker != content_tracker:
    with open(tracker_file_path, 'w') as f:
        f.write(new_content_tracker)
    print("Successfully updated import paths in src/tracker.py.")
else:
    print("No changes were needed for import paths in src/tracker.py.")

# --- 6. Patch `boxmot/trackers/common/appearance/__init__.py` for ReID.process ---
print("Patching `boxmot/trackers/common/appearance/__init__.py` for ReID.process changes...")
appearance_init_path = Path(boxmot_repo_path) / 'boxmot' / 'trackers' / 'common' / 'appearance' / '__init__.py'

if not appearance_init_path.exists():
    raise FileNotFoundError(f"Error: {appearance_init_path} not found. BoxMOT repository structure might be unexpected.")

with open(appearance_init_path, 'r') as f:
    content_appearance = f.readlines()

# Define the new resolve_batch_embeddings function content
new_resolve_batch_embeddings_lines = [
    "def resolve_batch_embeddings(",
    "    batch: DetectionBatch,",
    "    img: np.ndarray,",
    "    model=None,",
    "    enabled: bool = True,",
    "    boxes: np.ndarray | None = None,",
    "    placeholder_dim: int = 1,",
    "    placeholder_value: float = 1.0,",
    "    dtype=np.float32,",
    ") -> np.ndarray:",
    '    """Return embeddings aligned with a detection batch.',
    "", # This is an empty line in the docstring
    "    The order mirrors tracker update semantics: disabled appearance paths get",
    "    deterministic placeholders, precomputed embeddings are reused as-is, empty",
    "    batches avoid model calls, and only non-empty batches without precomputed",
    "    embeddings are passed through the ReID model.",
    '    """',
    "    if not enabled:",
    "        return placeholder_embeddings(",
    "            len(batch),",
    "            dim=placeholder_dim,",
    "            value=placeholder_value,",
    "            dtype=dtype,",
    "        )",
    "",
    "    if batch.embs is not None:",
    "        return batch.embs",
    "",
    "    if len(batch) == 0:",
    "        return placeholder_embeddings(",
    "            0,",
    "            dim=placeholder_dim,",
    "            value=placeholder_value,",
    "            dtype=dtype,",
    "        )",
    "",
    '    if model is None:',
    '        raise ValueError("A ReID model is required when embeddings are not provided")',
    "",
    "    # Pass the full image and detection bounding boxes to the ReID model's __call__ method.",
    "    # The ReID model internally handles cropping, preprocessing, and feature extraction.",
    "    return model(inputs=img, boxes=boxes)" # CHANGED FROM batch.xyxy to boxes
]

start_line_idx = -1
end_line_idx = -1
in_function = False

for i, line in enumerate(content_appearance):
    # Corrected regex to properly escape the parenthesis
    if re.match(r"def resolve_batch_embeddings" + re.escape("("), line):
        start_line_idx = i
        in_function = True
    # Corrected regex to properly escape the parenthesis
    elif in_function and re.match(r"def \w+" + re.escape("(") + r"|if __name__ == ", line):
        end_line_idx = i - 1
        while end_line_idx >= start_line_idx and content_appearance[end_line_idx].strip() == '':
            end_line_idx -= 1
        break
    elif in_function and i == len(content_appearance) - 1:
        end_line_idx = i
        break

if start_line_idx != -1 and end_line_idx != -1:
    original_indent = len(content_appearance[start_line_idx]) - len(content_appearance[start_line_idx].lstrip())
    indented_new_lines = []
    for j, new_line in enumerate(new_resolve_batch_embeddings_lines):
        if j == 0:
            indented_new_lines.append(' ' * original_indent + new_line.lstrip())
        else:
            indented_new_lines.append(' ' * original_indent + new_line)

    new_content_appearance_lines = (
        content_appearance[:start_line_idx] +
        [line + '\n' for line in indented_new_lines] +
        content_appearance[end_line_idx + 1:]
    )
    with open(appearance_init_path, 'w') as f:
        f.write("".join(new_content_appearance_lines))
    print("Successfully patched `resolve_batch_embeddings` function in appearance/__init__.py.")
else:
    print("Could not find `resolve_batch_embeddings` function for patching in appearance/__init__.py. The file might have changed unexpectedly.")

# --- 7. Patch `scripts/detect_track_export.py` for aggressive sys.path manipulation ---
print("Patching `scripts/detect_track_export.py` for robust sys.path prioritization...")
detect_track_export_path = PROJECT_CODE_DIR / 'scripts' / 'detect_track_export.py'

if not detect_track_export_path.exists():
    raise FileNotFoundError(f"Error: {detect_track_export_path} not found.")

with open(detect_track_export_path, 'r') as f:
    content_dte = f.read()

# Define the robust sys.path insertion block
line_to_insert_dte = f"""

# --- START BOXMOT PATH FORCE (Inserted by Colab Agent) ---
_boxmot_repo_path = '{boxmot_repo_path}'
# Explicitly remove any other boxmot installations from sys.path if they were loaded
# This is a robust way to ensure the editable install is used by the subprocess
_paths_to_remove = []
for p in list(sys.path): # Iterate over a copy as sys.path may be modified
    if 'boxmot' in p and _boxmot_repo_path not in p:
        _paths_to_remove.append(p)
for p in _paths_to_remove:
    sys.path.remove(p)
    print(f\"DEBUG: Removed '{{p}}' from sys.path to force correct boxmot usage.\")

# Ensure the cloned boxmot repo is at the very beginning of sys.path
if _boxmot_repo_path not in sys.path:
    sys.path.insert(0, _boxmot_repo_path)
    print(f\"DEBUG: Inserted '{{_boxmot_repo_path}}' to sys.path.\")

# Diagnostic prints to verify boxmot import source
try:
    import boxmot
    print(f\"DEBUG: boxmot imported from: {{boxmot.__file__}}\")
    from boxmot.trackers.common.appearance import resolve_batch_embeddings
    print(f\"DEBUG: resolve_batch_embeddings loaded from: {{resolve_batch_embeddings.__code__.co_filename}}\")
except Exception as e:
    print(f\"DEBUG: Error during boxmot inspection in detect_track_export: {{e}}\")
# --- END BOXMOT PATH FORCE ---

"""

# First, robustly remove any existing simple boxmot sys.path insertion to avoid duplication
pattern_to_remove_old_boxmot_block = re.compile(
    r"if\\s*'/tmp/boxmot_repo'\\s*not\\s*in\\s*sys\\.path:\\s*"
    r"sys\\.path\\.insert\\(0,\\s*'/tmp/boxmot_repo'\\)\\s*(?:#.*)?"
)
content_dte_after_removal = re.sub(pattern_to_remove_old_boxmot_block, "", content_dte)

# Define a more robust insertion point: after the PROJECT_ROOT sys.path insertion
insertion_point_pattern_dte = r"(sys\\.path\\.insert\\(0,\\s*str\\(PROJECT_ROOT\\)\\))"

new_content_dte = re.sub(insertion_point_pattern_dte, r"\\1" + line_to_insert_dte, content_dte_after_removal, count=1)

if new_content_dte != content_dte_after_removal:
    with open(detect_track_export_path, 'w') as f:
        f.write(new_content_dte)
    print("Successfully updated `scripts/detect_track_export.py` to prioritize the cloned boxmot repository.")
else:
    print("No changes were needed for `scripts/detect_track_export.py` path logic.")

print("---> Consolidated BoxMOT Setup & Patching Complete <---")

---> Starting Consolidated BoxMOT Setup & Patching <---
Uninstalling existing boxmot and ultralytics...
Uninstallation complete.
BoxMOT repository already cloned to /tmp/boxmot_repo. Skipping clone.
Installing boxmot in editable mode from /tmp/boxmot_repo...
BoxMOT installed in editable mode.
Reinstalling ultralytics...
Ultralytics reinstalled.
Patching `src/tracker.py` import paths...
No changes were needed for import paths in src/tracker.py.
Patching `boxmot/trackers/common/appearance/__init__.py` for ReID.process changes...
Successfully patched `resolve_batch_embeddings` function in appearance/__init__.py.
Patching `scripts/detect_track_export.py` for robust sys.path prioritization...
No changes were needed for `scripts/detect_track_export.py` path logic.
---> Consolidated BoxMOT Setup & Patching Complete <---
Please re-run Cell 7 or Cell 10 to test the fix.


### Cell 4 — Verify dataset and config

In [4]:
!python scripts/verify_dataset.py

VERIFYING PROJECT PATHS & DIRECTORIES CONFIGURATION

--- Checking Input Paths (Required to exist) ---

PROJECT_ROOT (Code repository root directory):
  Path: /content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Utilities/Football_MOT_Exporting
  [OK] Found and validated.

DATASET_ROOT (Root folder for the datasets):
  Path: /content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Datasets/OtherDatasets/AF-MOT
  [OK] Found and validated.

VIDEOS_DIR (Raw video clips folder):
  Path: /content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Datasets/OtherDatasets/AF-MOT/videos
  [OK] Found and validated.

ANNOT_ROOT (Ground-truth annotations folder):
  Path: /content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Datasets/OtherDatasets/AF-MOT/annotations
  [OK] Found and validated.

CONFIG_DIR (Configuration folder containing model and tracker YAMLs):
  Path: /content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Utilities/Football_MOT_Exporting/config

### Cell 5 — Verify Configuration (Smoke Test)

In [5]:
!python scripts/smoke_test_config.py

RUNNING SMOKE TEST: CONFIG AND PATH RESOLUTION
DATASET_ROOT:   /content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Datasets/OtherDatasets/AF-MOT
VIDEOS_DIR:     /content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Datasets/OtherDatasets/AF-MOT/videos
ANNOT_ROOT:     /content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Datasets/OtherDatasets/AF-MOT/annotations
PROJECT_ROOT:   /content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Utilities/Football_MOT_Exporting
MODEL_REGISTRY: /content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Utilities/Football_MOT_Exporting/configs/model_registry.yaml
BOTSORT_CONFIG: /content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Utilities/Football_MOT_Exporting/configs/football_botsort.yaml
EXPORT_DIR:     /content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Utilities/Football_MOT_Exporting/exports

Checking file and directory presence:
  - PROJECT_ROOT exists: True
  - MODEL_REGISTRY exist

### Cell 6 — Verify Model Loader (Smoke Test)

In [6]:
!python scripts/smoke_test_model_load.py

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
RUNNING SMOKE TEST: MODEL LOADING
Available registry models: ['rtdetrx_player_best', 'yolo26x_player_best']

Checking model: 'rtdetrx_player_best'
  - Family:       rtdetr
  - Variant:      rtdetr-x
  - Weights path: /content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Datasets/OtherDatasets/Object_Detection/Models/YOLO/Output/RTDETR/weights/best.pt
  - Weights file exists: True
  - Loading model key 'rtdetrx_player_best'...
  - Successfully loaded model. Family: rtdetr

Checking model: 'yolo26x_player_best'
  - Family:       yolo
  - Variant:      yolo26x
  - Weights path: /content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Datasets/OtherDatasets/Object_Detec

### Cell 7 — Quick Single-Frame Smoke Test (Optional)

In [30]:
from configs.project_config import VIDEOS_DIR, EXPORT_DIR

sample_video = VIDEOS_DIR / 'SWide008.mp4'
out_zip = EXPORT_DIR / 'smoke_test_output_cvat.zip'

# Construct the shell command as a Python f-string to ensure variable interpolation
# before passing to the shell. This will correctly resolve the Paths to strings.
command = f"""PYTHONPATH=/tmp/boxmot_repo:$PYTHONPATH python scripts/detect_track_export.py \
    --video "{str(sample_video)}" \
    --model-key rtdetrx_player_best \
    --out "{str(out_zip)}" \
    --conf 0.64 \
    --max-frames 1 \
    --save-preview"""

# Execute the constructed command
!{command}

DEBUG: boxmot imported from: /tmp/boxmot_repo/boxmot/__init__.py
DEBUG: resolve_batch_embeddings loaded from: /tmp/boxmot_repo/boxmot/trackers/common/appearance/__init__.py
DEBUG: boxmot imported from: /tmp/boxmot_repo/boxmot/__init__.py
DEBUG: resolve_batch_embeddings loaded from: /tmp/boxmot_repo/boxmot/trackers/common/appearance/__init__.py
[2026-07-15 15:17:29] [INFO] Loading model info for key: rtdetrx_player_best...
[2026-07-15 15:17:31] [INFO] Initializing tracker...
INFO     BoxMOT v19.0.0 🚀 Python-3.12.13 torch-2.11.0+cpuCPU                   
INFO     /tmp/boxmot_repo/models/osnet_x0_25_msmt17.pt                          
INFO     [PID 20423] Found existing ReID weights at                             
         /tmp/boxmot_repo/models/osnet_x0_25_msmt17.pt; skipping download.      
INFO     Loaded pretrained weights from                                         
         /tmp/boxmot_repo/models/osnet_x0_25_msmt17.pt                          
[FootballMOT] Successfully instantia

### Cell 8 — Export One Sample Video to CVAT MOT 1.1

Runs player detection and BoT-SORT tracking on a single football video and generates the CVAT MOT 1.1 ZIP export.

In [ ]:
from configs.project_config import VIDEOS_DIR, EXPORT_DIR

# sample_video = VIDEOS_DIR / 'SWide008'
# sample_video = VIDEOS_DIR / 'JetSweep_1'
sample_video = VIDEOS_DIR / 'Counter_1'
out_zip = EXPORT_DIR / f"{sample_video.stem}_cvat_mot.zip"

!python scripts/detect_track_export.py \
    --video "$sample_video" \
    --model-config configs/model_registry.yaml \
    --model-key rtdetrx_player_best \
    --tracker-config configs/football_botsort.yaml \
    --out "$out_zip" \
    --conf 0.64 \
    --save-preview

### Cell 9 — MOT Evaluation

Evaluates predicted tracks from a CVAT MOT 1.1 ZIP file against the ground truth dataset annotation `gt.txt`.

In [ ]:
from configs.project_config import ANNOT_ROOT, EXPORT_DIR, get_annot_dir_name

# sequence_name = "SWide008"
sequence_name = 'JetSweep_1'
gt_path = ANNOT_ROOT / get_annot_dir_name(sequence_name) / "gt" / "gt.txt"
pred_zip = EXPORT_DIR / f"{sequence_name}_cvat_mot.zip"

# Save evaluation files
metrics_json = EXPORT_DIR / "metrics" / f"{sequence_name}_metrics.json"
metrics_md = EXPORT_DIR / "metrics" / f"{sequence_name}_metrics.md"

!python scripts/evaluate_mot.py \
    --gt "$gt_path" \
    --pred "$pred_zip" \
    --sequence-name "$sequence_name" \
    --out-json "$metrics_json" \
    --out-md "$metrics_md"

### Cell 10 — Batch Export

Runs tracking and creates CVAT export over the `ToBeTracked.csv`.

In [31]:
from pathlib import Path
import subprocess

# Define the paths directly to bypass Google Drive sync delay
TO_BE_TRACKED = Path('/content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Datasets/AmericanFootballDataset/videos')
EXPORT_DIR = Path('/content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Utilities/Football_MOT_Exporting/exports')

# 1. Verify clips directory exists before running
if not TO_BE_TRACKED.exists():
    raise FileNotFoundError(
        f"The clips directory (TO_BE_TRACKED) does not exist at: {TO_BE_TRACKED.resolve()}\n\n"
        f"Please verify that your Google Drive is mounted correctly (Cell 1)."
    )

# 2. Resolve the CSV path
csv_path = Path("/content/ToBeTracked.csv")
if not csv_path.exists():
    csv_path = Path("ToBeTracked.csv")

if not csv_path.exists():
    try:
        from google.colab import files
        print('Please upload your ToBeTracked.csv file:')
        uploaded = files.upload()
        if uploaded:
            csv_filename = list(uploaded.keys())[0]
            csv_path = Path(csv_filename)
            print(f'Uploaded successfully: {csv_path}')
        else:
            print('No file uploaded. Proceeding without CSV filtering.')
            csv_path = None
    except ImportError:
        print('Not in Google Colab environment. Proceeding without CSV filtering.')
        csv_path = None
else:
    print(f'Using existing CSV file: {csv_path.resolve()}')

batch_out_dir = EXPORT_DIR / "cvat_mot_tracks" / "batch" / "Counter_Reclips"   # Change the end folder for a new play

cmd = [
    "python", "scripts/batch_export.py",
    "--model-config", "configs/model_registry.yaml",
    "--model-key", "rtdetrx_player_best",
    "--tracker-config", "configs/football_botsort.yaml",
    "--clips-dir", str(TO_BE_TRACKED),
    "--out-dir", str(batch_out_dir),
    "--conf", "0.64",
    "--save-preview",
    "--write-summary"
]

if csv_path and csv_path.exists():
    cmd.extend(["--csv-list", str(csv_path)])

print(f"Executing: {' '.join(cmd)}")

# 3. Run and capture output to print if it fails
result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode != 0:
    print("\n=== SUBPROCESS FAILED ===")
    print("STDOUT:")
    print(result.stdout)
    print("STDERR:")
    print(result.stderr)
    raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout, stderr=result.stderr)
else:
    print("\n=== SUBPROCESS COMPLETED SUCCESSFULLY ===")
    print(result.stdout)


Using existing CSV file: /content/ToBeTracked.csv
Executing: python scripts/batch_export.py --model-config configs/model_registry.yaml --model-key rtdetrx_player_best --tracker-config configs/football_botsort.yaml --clips-dir /content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Datasets/AmericanFootballDataset/videos --out-dir /content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Utilities/Football_MOT_Exporting/exports/cvat_mot_tracks/batch/Counter_Reclips --conf 0.64 --save-preview --write-summary --csv-list /content/ToBeTracked.csv

=== SUBPROCESS COMPLETED SUCCESSFULLY ===
DEBUG: boxmot imported from: /tmp/boxmot_repo/boxmot/__init__.py
DEBUG: resolve_batch_embeddings loaded from: /tmp/boxmot_repo/boxmot/trackers/common/appearance/__init__.py
DEBUG: boxmot imported from: /tmp/boxmot_repo/boxmot/__init__.py
DEBUG: resolve_batch_embeddings loaded from: /tmp/boxmot_repo/boxmot/trackers/common/appearance/__init__.py
[2026-07-15 15:20:52] [INFO] Found 10 videos in /co

### Cell 11 — Summarize Batch Results

Aggregates the batch tracking outputs and prints a summary report.

In [ ]:
from configs.project_config import EXPORT_DIR

summary_json = EXPORT_DIR / "cvat_mot_tracks" / "batch" / "batch_tracking_summary.json"

!python scripts/summarize_exports.py \
    --summary "$summary_json"

### Cell 12 — Re-render Previews with Dynamic / Adjusted Bounding Boxes (Fast, No MOT Model Re-run)

Use this cell when you want to re-render preview videos with adjusted, smaller, or resolution-scaled bounding boxes and track ID text (e.g., for YouTube clips or lower-resolution videos).

This script reads existing `*_cvat_mot.zip` tracking exports and original video clips, re-rendering preview MP4s **without re-running object detection or MOT tracking**.

In [ ]:
from pathlib import Path
import subprocess

# 1. Define folder paths
CLIPS_DIR = Path('/content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Datasets/AmericanFootballDataset/videos')
ZIPS_DIR = Path('/content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Utilities/Football_MOT_Exporting/exports/cvat_mot_tracks/batch/Counter_Reclips')
OUT_DIR = Path('/content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Utilities/Football_MOT_Exporting/exports/previews_adjusted')

# Fallback if specific batch folder doesn't exist
if not ZIPS_DIR.exists():
    ZIPS_DIR = Path('/content/drive/MyDrive/UCF/Courses/Spring2026/IndependentStudy/Utilities/Football_MOT_Exporting/exports/cvat_mot_tracks')

if not CLIPS_DIR.exists():
    raise FileNotFoundError(
        f"The clips directory (CLIPS_DIR) does not exist at: {CLIPS_DIR.resolve()}\n"
        f"Please verify that Google Drive is mounted correctly in Cell 1."
    )

# 2. Check for optional CSV filtering list
csv_path = Path("/content/ToBeTracked.csv")
if not csv_path.exists():
    csv_path = Path("ToBeTracked.csv")

# 3. Build command
cmd = [
    "python", "scripts/batch_rerender_previews.py",
    "--zips-dir", str(ZIPS_DIR),
    "--clips-dir", str(CLIPS_DIR),
    "--out-dir", str(OUT_DIR),
    "--overwrite"
]

# Optional explicit overrides (uncomment to force fixed values instead of auto-resolution scaling):
# cmd.extend(["--font-scale", "0.35", "--box-thickness", "1"])

if csv_path and csv_path.exists():
    print(f"Using CSV file for selective re-rendering: {csv_path.resolve()}")
    cmd.extend(["--csv-list", str(csv_path)])
else:
    print(f"No CSV list found. Auto-scanning directory: {ZIPS_DIR.resolve()}")

print(f"Executing: {' '.join(cmd)}")

# 4. Execute re-rendering
result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode != 0:
    print("\n=== RE-RENDER SUBPROCESS FAILED ===")
    print("STDOUT:")
    print(result.stdout)
    print("STDERR:")
    print(result.stderr)
    raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout, stderr=result.stderr)
else:
    print("\n=== RE-RENDER SUBPROCESS COMPLETED SUCCESSFULLY ===")
    print(result.stdout)
